In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [ ]:
def random_clifford_t_block(qc, q1, q2, add_t, p_t=0.2):
    for q in [q1, q2]:
        r = np.random.randint(4)
        if r == 0:
            qc.h(q)
        elif r == 1:
            qc.s(q)
        elif r == 2:
            qc.h(q)
            qc.s(q)

    if np.random.rand() < 0.5:
        qc.cx(q1, q2)
    else:
        qc.cx(q2, q1)

    if add_t:
        if np.random.rand() < p_t:
            qc.t(q1)
        if np.random.rand() < p_t:
            qc.t(q2)


def build_circuit_2d(Lx, Ly, d, add_t, p_t=0.2):
    N = Lx * Ly
    qc = QuantumCircuit(N, N)

    def idx(x, y):
        return y * Lx + x

    for layer in range(d):
        direction = layer % 2
        offset = (layer // 2) % 2

        gates_added = False
        if direction == 0:
            for y in range(Ly):
                for x in range(offset, Lx - 1, 2):
                    random_clifford_t_block(qc, idx(x, y), idx(x + 1, y), add_t=add_t, p_t=p_t)
                    gates_added = True
        else:
            for x in range(Lx):
                for y in range(offset, Ly - 1, 2):
                    random_clifford_t_block(qc, idx(x, y), idx(x, y + 1), add_t=add_t, p_t=p_t)
                    gates_added = True

        if gates_added:
            qc.barrier()

    qc.measure(range(N), range(N))
    return qc


def projected_collision_prob(counts, n_A):
    total_shots = sum(counts.values())

    # group B-outcome counts by A outcome
    ab_counts = defaultdict(lambda: defaultdict(int))
    a_counts = defaultdict(int)

    for bitstring, count in counts.items():
        bitstring = bitstring.replace(' ', '')
        a_bits = bitstring[-n_A:]
        b_bits = bitstring[:-n_A]
        ab_counts[a_bits][b_bits] += count
        a_counts[a_bits] += count

    cp = 0
    for a_bits, b_dict in ab_counts.items():
        p_a = a_counts[a_bits] / total_shots
        n_a = a_counts[a_bits]
        cond_cp = sum((c / n_a) ** 2 for c in b_dict.values())
        cp += p_a * cond_cp

    return cp


: 

In [ ]:
# Vary depth, fixed N
L_values = [6]
depth_values = list(range(2, 14))
samples_per_n = 10
shots = 2000
add_t = True
output_dir = "results/projected_cp"
os.makedirs(output_dir, exist_ok=True)

sim = AerSimulator(
    method="matrix_product_state",
    mps_sample_measure_algorithm="mps_apply_measure",
    matrix_product_state_max_bond_dimension=64,
    matrix_product_state_truncation_threshold=1e-8,
    max_parallel_threads=0,
    mps_omp_threads=0
)

for L in L_values:
    N = L * L
    n_A = N // 2
    n_B = N - n_A
    avg_prob, std_prob, completed_depths = [], [], []

    for d in depth_values:
        print(f"d={d}, N={N}, n_A={n_A}, n_B={n_B}")
        vals = []

        for _ in range(samples_per_n):
            qc = build_circuit_2d(L, L, d, add_t, p_t=0.15)
            tqc = transpile(qc, basis_gates=['h', 's', 'sdg', 't', 'tdg', 'cx', 'measure'], optimization_level=0)
            counts = sim.run(tqc, shots=shots).result().get_counts()
            vals.append((2 ** n_B) * projected_collision_prob(counts, n_A))

        avg_prob.append(np.mean(vals))
        std_prob.append(np.std(vals))
        completed_depths.append(d)

        # Save cumulative plot after every depth
        fig, ax = plt.subplots()
        ax.errorbar(completed_depths, avg_prob, yerr=std_prob, marker='o', capsize=4, label=f"N={N}")
        ax.set_xlabel("depth")
        ax.set_ylabel(r"$2^{n_B} \sum_x p(x) \sum_y p(y|x)^2$")
        ax.set_yscale('log')
        ax.set_title(f"2D Brickwork: Projected Ensemble CP  (N={N}, up to depth={d})")
        ax.grid(True, which='both', alpha=0.3)
        ax.legend()
        fig.savefig(os.path.join(output_dir, f"projected_cp_N{N}_upto_d{d}.png"), dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"  saved projected_cp_N{N}_upto_d{d}.png")

# Show final plot inline
plt.errorbar(completed_depths, avg_prob, yerr=std_prob, marker='o', capsize=4, label=f"N={N}")
plt.xlabel("depth")
plt.ylabel(r"$2^{n_B} \sum_x p(x) \sum_y p(y|x)^2$")
plt.yscale('log')
plt.title(f"2D Brickwork: Projected Ensemble CP  (varying depth, fixed N={N})")
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()


d=2, N=36, n_A=18, n_B=18
  saved projected_cp_N36_upto_d2.png
d=3, N=36, n_A=18, n_B=18
  saved projected_cp_N36_upto_d3.png
d=4, N=36, n_A=18, n_B=18
  saved projected_cp_N36_upto_d4.png
d=5, N=36, n_A=18, n_B=18
  saved projected_cp_N36_upto_d5.png
d=6, N=36, n_A=18, n_B=18
